In [ ]:
# =====================================================================
# ASSIGNMENT 1 (GENERIC): HTML / WEBSITE  ->  JSON  ->  DataFrame  ->  wrangling -> features -> Min-Max -> plots -> CSV
# HOW TO USE: only edit the "STEP 0: CONFIG" block. Everything below it runs automatically.
# Use browser "Inspect element" (F12) to find the CSS selectors for the config.
# =====================================================================
import json, re
from urllib.parse import urljoin
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
sns.set_theme(style="whitegrid")

# ---------------------- STEP 0: CONFIG (EDIT THIS) ----------------------
# (a) WHERE IS THE DATA?
#   - local file  -> SOURCES = ["my_page.html"]
#   - one website -> SOURCES = ["https://books.toscrape.com/catalogue/page-1.html"]
#   - many pages  -> SOURCES = [f"https://books.toscrape.com/catalogue/page-{i}.html" for i in range(1, 51)]
SOURCES = ["my_page.html"]

# (b) ONE REPEATING BLOCK (one book / one job / one event / one movie) as a CSS selector.
#   examples: "article.product_pod" | "div.job-card" | "div.event" | "li.ipc-metadata-list-summary-item"
CARD = "div.card"

# (c) FIELDS TO EXTRACT INSIDE ONE BLOCK:  "column_name": selector
#   "title": "h3"                      -> text of the first <h3> in the block
#   "url":   ("a", "href")             -> an attribute (href / src / title / class ...) instead of text
#   "skills": ("div.skills span", "list") -> many tags collected as a Python list
#   A missing tag is stored as None automatically (no crash).
FIELDS = {
    "title":  "h3",
    "price":  "p.price",
    "rating": ("p.rating", "class"),   # class attr gives e.g. ['star-rating','Three']
    "url":    ("a", "href"),
}

# (d) OPTIONAL: open every item's detail page to get more attributes (category, UPC, description ...).
#   Put the name of the column holding the link in DETAIL_LINK_COL and the selectors in DETAIL_FIELDS.
#   Leave DETAIL_LINK_COL = None to skip. (Slow: one request per item.)
DETAIL_LINK_COL = None            # e.g. "url"
DETAIL_FIELDS   = {}              # e.g. {"category": "ul.breadcrumb li:nth-of-type(3)", "upc": "table tr:nth-of-type(1) td"}
MAX_ITEMS       = None            # e.g. 500 to stop after 500 items

# (e) WRANGLING SETTINGS (use [] / {} when not needed)
NUMERIC_COLS = ["price"]          # text -> number (keeps first number: '£51.77' -> 51.77, '3 LPA' -> 3)
RANGE_COLS   = []                 # text with a range '6 - 8 LPA' -> adds <col>_min, <col>_max, <col>_avg
RATING_COLS  = {"rating": {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}}  # word -> number mapping
DATE_COLS    = {}                 # {"posted": "%d %B %Y"}  (also remove prefixes like 'Posted:' via PREFIX_REMOVE)
PREFIX_REMOVE = ["Posted:"]       # words to delete from all text before parsing
CASE_FIX_COLS = []                # columns to make Title Case (fixes 'hyderabad' vs 'Hyderabad')
NORMALIZE_COLS = ["price"]        # columns for Min-Max normalization (needs to be numeric after cleaning)
OUT_CSV = "cleaned_dataset.csv"
# -------------------------------------------------------------------------

# ---------------------- STEP 1: HTML -> JSON (semi-structured) ----------------------
def get_soup(src):
    if src.startswith("http"):                       # website
        r = requests.get(src, timeout=20); r.encoding = "utf-8"
        return BeautifulSoup(r.text, "html.parser")
    with open(src, encoding="utf-8") as f:           # local html file
        return BeautifulSoup(f, "html.parser")

def extract(parent, spec):
    """Safe extractor: returns None if the tag is missing."""
    if isinstance(spec, str):                        # plain text
        el = parent.select_one(spec);  return el.get_text(strip=True) if el else None
    sel, mode = spec
    if mode == "list":                               # many tags -> list
        return [e.get_text(strip=True) for e in parent.select(sel)]
    el = parent.select_one(sel)
    if el is None: return None
    if mode == "text": return el.get_text(strip=True)
    val = el.get(mode)                               # attribute
    return " ".join(val) if isinstance(val, list) else val      # class -> 'star-rating Three'

records = []
for src in SOURCES:
    soup = get_soup(src)
    for card in soup.select(CARD):
        rec = {k: extract(card, v) for k, v in FIELDS.items()}
        if DETAIL_LINK_COL and rec.get(DETAIL_LINK_COL):                  # optional detail page
            rec[DETAIL_LINK_COL] = urljoin(src, rec[DETAIL_LINK_COL])      # relative -> absolute url
            dsoup = get_soup(rec[DETAIL_LINK_COL])
            rec.update({k: extract(dsoup, v) for k, v in DETAIL_FIELDS.items()})
        records.append(rec)
        if MAX_ITEMS and len(records) >= MAX_ITEMS: break
    if MAX_ITEMS and len(records) >= MAX_ITEMS: break

print("Records scraped:", len(records))
with open("data.json", "w", encoding="utf-8") as f:                       # save JSON
    json.dump(records, f, indent=4, ensure_ascii=False)
with open("data.json", encoding="utf-8") as f:                            # read it back
    data = json.load(f)

# ---------------------- STEP 2: JSON -> DataFrame (structured) ----------------------
df = pd.DataFrame(data)
print(df.head()); print(); df.info()
print("\nMissing values:\n", df.isnull().sum())

# ---------------------- STEP 3: DATA WRANGLING ----------------------
list_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, list)).any()]
text_cols = [c for c in df.columns if c not in list_cols and not pd.api.types.is_numeric_dtype(df[c])]   # all text columns

# 3.1 Fix text FIRST (strip spaces, remove prefixes, fix case) -- before removing duplicates
for c in text_cols:
    df[c] = df[c].str.strip()
    for p in PREFIX_REMOVE: df[c] = df[c].str.replace(p, "", regex=False).str.strip()
for c in CASE_FIX_COLS: df[c] = df[c].str.title()

# 3.2 Duplicates (list columns can't be hashed, so leave them out)
print("Duplicates found:", df.duplicated(subset=[c for c in df.columns if c not in list_cols]).sum())
df = df.drop_duplicates(subset=[c for c in df.columns if c not in list_cols]).reset_index(drop=True)

# 3.3 Convert types
for c, mp in RATING_COLS.items():                                          # 'Three' -> 3 (also finds the word inside 'star-rating Three')
    df[c] = df[c].apply(lambda s: next((v for k, v in mp.items() if s and k in s), np.nan))
def first_num(s):
    m = re.search(r"\d+(?:,\d+)*(?:\.\d+)?", str(s)) if s is not None else None
    return float(m.group().replace(",", "")) if m else np.nan
for c in NUMERIC_COLS: df[c] = df[c].apply(first_num)
for c in RANGE_COLS:                                                       # '6 - 8 LPA' -> min 6, max 8, avg 7
    nums = df[c].apply(lambda s: [float(x) for x in re.findall(r"\d+(?:\.\d+)?", str(s))] if s else [])
    df[c + "_min"] = nums.apply(lambda l: l[0] if l else np.nan)
    df[c + "_max"] = nums.apply(lambda l: l[-1] if l else np.nan)
    df[c + "_avg"] = (df[c + "_min"] + df[c + "_max"]) / 2
for c, fmt in DATE_COLS.items(): df[c] = pd.to_datetime(df[c], format=fmt, errors="coerce")

# 3.4 Missing values: numbers -> median, text -> 'Not Specified'  (change if the question says otherwise)
num_cols = df.select_dtypes(include="number").columns.tolist()
for c in num_cols: df[c] = df[c].fillna(df[c].median())
for c in [c for c in text_cols if c in df.columns]: df[c] = df[c].fillna("Not Specified")

# ---------------------- STEP 4: FEATURE ENGINEERING (>= 3 new columns) ----------------------
# Automatic features below. REPLACE / ADD your own that fit the question, e.g.
#   df["price_per_rating"] = df["price"] / df["rating"]      df["is_free"] = df["fee"] == 0
first_text = next((c for c in text_cols if c in df.columns and c not in ("url",)), None)
main_num = num_cols[0] if num_cols else None
if first_text: df[first_text + "_length"] = df[first_text].astype(str).str.len()          # text length
if main_num:                                                                                # low / medium / high by tercile
    df[main_num + "_category"] = pd.qcut(df[main_num], 3, labels=["Low", "Medium", "High"], duplicates="drop")
for c in list_cols: df[c + "_count"] = df[c].apply(len)                                     # items in a list column
for c in DATE_COLS: df[c + "_days_ago"] = (pd.Timestamp.today().normalize() - df[c]).dt.days  # age in days
if len(num_cols) >= 2:                                                                      # ratio of first two numeric columns
    df[f"{num_cols[0]}_per_{num_cols[1]}"] = (df[num_cols[0]] / df[num_cols[1]].replace(0, np.nan)).round(2)

# ---------------------- STEP 5: MIN-MAX NORMALIZATION ----------------------
# x_norm = (x - min) / (max - min)  -> scales to 0..1.
# JUSTIFICATION (write this in the report): numeric columns have different scales (e.g. price 10-60 vs rating 1-5);
# Min-Max brings them to the same 0-1 range so no feature dominates in distance-based methods (KNN, K-Means),
# and it keeps the shape of the distribution. Weakness: sensitive to outliers.
for c in NORMALIZE_COLS:
    df[c + "_norm"] = ((df[c] - df[c].min()) / (df[c].max() - df[c].min())).round(3)
print(df.head())

# ---------------------- STEP 6: VISUALIZATIONS (auto, >= 5 plots) ----------------------
cat_cols = [c for c in df.columns if c not in list_cols and not pd.api.types.is_numeric_dtype(df[c])
            and not pd.api.types.is_datetime64_any_dtype(df[c]) and 1 < df[c].nunique() <= 30]   # short text/category columns
num_plot = [c for c in df.select_dtypes(include="number").columns if not c.endswith("_norm")]
plots = []
if num_plot:  plots.append(lambda ax: (ax.hist(df[num_plot[0]], bins=20, color="coral", edgecolor="black"), ax.set_title(f"Distribution of {num_plot[0]}")))
if len(num_plot) > 1: plots.append(lambda ax: (ax.hist(df[num_plot[1]], bins=20, color="gold", edgecolor="black"), ax.set_title(f"Distribution of {num_plot[1]}")))
if cat_cols:  plots.append(lambda ax: (df[cat_cols[0]].value_counts().head(10).plot(kind="barh", ax=ax, color="seagreen"), ax.set_title(f"Top values of {cat_cols[0]}")))
if cat_cols:  plots.append(lambda ax: (df[cat_cols[-1]].value_counts().head(8).plot(kind="pie", autopct="%1.0f%%", ax=ax), ax.set_title(f"Share of {cat_cols[-1]}"), ax.set_ylabel("")))
if cat_cols and num_plot: plots.append(lambda ax: (sns.boxplot(data=df, x=cat_cols[-1], y=num_plot[0], ax=ax), ax.set_title(f"{num_plot[0]} by {cat_cols[-1]}")))
if len(num_plot) > 1: plots.append(lambda ax: (sns.scatterplot(data=df, x=num_plot[0], y=num_plot[1], ax=ax), ax.set_title(f"{num_plot[0]} vs {num_plot[1]}")))
if cat_cols and num_plot: plots.append(lambda ax: (df.groupby(cat_cols[0])[num_plot[0]].mean().sort_values().tail(10).plot(kind="barh", ax=ax, color="mediumpurple"), ax.set_title(f"Avg {num_plot[0]} per {cat_cols[0]}")))
rows = (len(plots) + 2) // 3
fig, axes = plt.subplots(rows, 3, figsize=(18, 5 * rows)); axes = np.array(axes).ravel()
for p, ax in zip(plots, axes): p(ax)
for ax in axes[len(plots):]: ax.axis("off")
plt.tight_layout(); plt.show()
if len(num_plot) > 1:                                                         # correlation heatmap
    plt.figure(figsize=(7, 5)); sns.heatmap(df[num_plot].corr(), annot=True, cmap="coolwarm", fmt=".2f"); plt.title("Correlation"); plt.show()

# ---------------------- STEP 7: SAVE CSV + INSIGHTS ----------------------
out = df.copy()
for c in list_cols: out[c] = out[c].apply(lambda x: ", ".join(x))            # list -> text so CSV can store it
out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Saved", OUT_CSV, out.shape)

print("\nINSIGHTS (copy & edit into your report)")
print("1. Total unique records:", len(df))
if main_num:
    print(f"2. {main_num}: mean={df[main_num].mean():.2f}, median={df[main_num].median():.2f}, min={df[main_num].min()}, max={df[main_num].max()}")
    if first_text: print(f"3. Highest {main_num}:", df.loc[df[main_num].idxmax(), first_text])
if cat_cols:
    print(f"4. Most common {cat_cols[0]}:", df[cat_cols[0]].value_counts().idxmax(), "->", df[cat_cols[0]].value_counts().max())
    if main_num: print(f"5. Highest avg {main_num} in {cat_cols[0]}:", df.groupby(cat_cols[0])[main_num].mean().idxmax())
if len(num_plot) > 1:
    c = df[num_plot].corr().abs().where(~np.eye(len(num_plot), dtype=bool)).stack().idxmax()
    print("6. Strongest correlation:", c, round(df[list(c)].corr().iloc[0, 1], 2))

In [ ]:
# =====================================================================
# ASSIGNMENT 2 (GENERIC): REGRESSION on ANY CSV  ->  EDA -> preprocessing -> Models A/B/C -> feature selection (RFE)
# HOW TO USE: edit only "STEP 0: CONFIG". Needs a CSV with a numeric target column (price, value, salary ...).
# =====================================================================
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---------------------- STEP 0: CONFIG (EDIT THIS) ----------------------
CSV_PATH  = None            # e.g. "Housing.csv"  (None = use sklearn's California Housing demo data)
TARGET    = "MedHouseVal"   # name of the column you want to predict (CHANGE for your CSV)
DROP_COLS = []              # useless columns (ids, names, dates) e.g. ["id", "Unnamed: 0"]
CLIP_OUTLIERS = True        # True = treat outliers with IQR clipping (winsorization)
TEST_SIZE = 0.20
N_A, N_B  = 3, 5            # Model A = top N_A correlated features, Model B = top N_B (as the question says)
N_RFE     = 5               # features to keep in the RFE feature-selection part
# -------------------------------------------------------------------------

# ---------------------- STEP 1: LOAD + EDA ----------------------
if CSV_PATH is None:
    from sklearn.datasets import fetch_california_housing
    df = fetch_california_housing(as_frame=True).frame
else:
    df = pd.read_csv(CSV_PATH)
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

print("Shape:", df.shape); print("\nFirst rows:\n", df.head())
print("\nData types:\n", df.dtypes); print("\nSummary statistics:\n", df.describe())
print("\nMissing values:\n", df.isnull().sum()); print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# yes/no style or text columns -> numbers (so the correlation / regression can use them)
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
for c in cat_cols:
    if set(df[c].dropna().unique()) <= {"yes", "no"}: df[c] = df[c].map({"yes": 1, "no": 0})  # binary columns
df = pd.get_dummies(df, columns=[c for c in cat_cols if df[c].dtype == object or str(df[c].dtype) == "str"], drop_first=True)
df = df.astype({c: float for c in df.select_dtypes("bool").columns})                          # True/False -> 1/0

num_df = df.select_dtypes(include="number")
num_df.hist(figsize=(15, 10), bins=30, edgecolor="black"); plt.suptitle("Histograms"); plt.tight_layout(); plt.show()   # histograms
plt.figure(figsize=(14, 5)); sns.boxplot(data=(num_df - num_df.mean()) / num_df.std()); plt.xticks(rotation=45)          # box plots (standardised so they fit on one axis)
plt.title("Box plots (standardised)"); plt.tight_layout(); plt.show()
corr = num_df.corr()
top_feat = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).index[0]
plt.figure(figsize=(6, 4)); plt.scatter(num_df[top_feat], num_df[TARGET], alpha=0.3)                                      # scatter: best feature vs target
plt.xlabel(top_feat); plt.ylabel(TARGET); plt.title(f"{top_feat} vs {TARGET}"); plt.show()
plt.figure(figsize=(11, 8)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm"); plt.title("Correlation heatmap"); plt.show()
print("Correlation with target:\n", corr[TARGET].sort_values(ascending=False))

# ---------------------- STEP 2: OUTLIERS + MISSING + SPLIT + SCALE ----------------------
Q1, Q3 = num_df.quantile(0.25), num_df.quantile(0.75); IQR = Q3 - Q1
low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
print("\nOutliers per column (IQR rule):\n", ((num_df < low) | (num_df > up)).sum())
df_clean = num_df.copy()
if CLIP_OUTLIERS:                                                    # winsorization: cap values at the IQR fences
    for c in df_clean.columns: df_clean[c] = df_clean[c].clip(low[c], up[c])
df_clean = df_clean.fillna(df_clean.median())                        # missing numeric -> median

X, y = df_clean.drop(columns=TARGET), df_clean[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=42)
scaler = StandardScaler()                                            # fit on TRAIN only, transform both (avoids data leakage)
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)
print("Train:", X_train.shape, "Test:", X_test.shape)

# ---------------------- STEP 3: MODELS A / B / C ----------------------
# Correlation is computed on TRAIN data only. A = top few correlated features, B = more, C = everything.
tr = X_train.copy(); tr[TARGET] = y_train
corr_abs = tr.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
feature_sets = {"Model A": corr_abs.head(N_A).index.tolist(),
                "Model B": corr_abs.head(N_B).index.tolist(),
                "Model C": X_train.columns.tolist()}

def evaluate(name, feats, model_cls=LinearRegression):
    m = model_cls(); t0 = time.perf_counter(); m.fit(X_train[feats], y_train); t = time.perf_counter() - t0   # training time
    p = m.predict(X_test[feats]); mse = mean_squared_error(y_test, p)
    return m, {"Model": name, "Features": len(feats), "MAE": mean_absolute_error(y_test, p), "MSE": mse,
               "RMSE": np.sqrt(mse), "R2": r2_score(y_test, p), "Train Time (s)": t}

rows = []
for name, feats in feature_sets.items():
    print(name, "->", feats); rows.append(evaluate(name, feats)[1])
res = pd.DataFrame(rows); print("\nComparative table:\n", res.round(6).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, m_ in zip(ax, ["RMSE", "R2", "Train Time (s)"]):
    sns.barplot(data=res, x="Model", y=m_, ax=a); a.set_title(m_)
plt.tight_layout(); plt.show()
best = res.loc[res["R2"].idxmax()]
print(f"\nBest model: {best['Model']} (R2={best['R2']:.4f}, RMSE={best['RMSE']:.4f}, time={best['Train Time (s)']:.5f}s)")
# WRITE IN REPORT: more features -> usually higher R2 / lower RMSE but slightly more training time;
# if A/B are almost as good as C, the extra features add computational cost for very little accuracy.

# ---------------------- STEP 4: FEATURE SELECTION with RFE + comparison with all features ----------------------
rfe = RFE(LinearRegression(), n_features_to_select=min(N_RFE, X_train.shape[1])).fit(X_train, y_train)
sel = X_train.columns[rfe.support_].tolist(); print("\nRFE selected features:", sel)
coef = pd.Series(LinearRegression().fit(X_train, y_train).coef_, index=X_train.columns)
print("Coefficient importance (|coef|):\n", coef.abs().sort_values(ascending=False))   # alternative selection method
cmp_ = pd.DataFrame([evaluate("All features", X_train.columns.tolist())[1], evaluate("RFE selected", sel)[1]])
print("\nAll vs Selected:\n", cmp_.round(6).to_string(index=False))
d = cmp_.loc[1, "R2"] - cmp_.loc[0, "R2"]
print("Conclusion:", "selection IMPROVES accuracy" if d > 1e-3 else "selection DEGRADES accuracy" if d < -1e-3 else "selection MAINTAINS accuracy",
      f"(R2 change = {d:+.4f}). Fewer features = simpler, faster model.")

In [ ]:
# =====================================================================
# ASSIGNMENT 3 (GENERIC): PART A classification | PART B PCA & SVD | PART C Ridge/Lasso/ElasticNet
# HOW TO USE: edit only "STEP 0: CONFIG". Part A+B use a CLASSIFICATION csv/xlsx (binary target, e.g. churn 0/1 or Yes/No).
# Part C uses a separate REGRESSION csv (numeric target, e.g. house price).
# =====================================================================
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,
                             precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay,
                             mean_squared_error, mean_absolute_error, r2_score)
sns.set_theme(style="whitegrid")

# ---------------------- STEP 0: CONFIG (EDIT THIS) ----------------------
# ---- Part A / B : classification data ----
CLS_PATH   = None        # "Telco_customer_churn.xlsx" or "data.csv"  (None = sklearn breast-cancer demo)
CLS_TARGET = "target"    # column to predict, e.g. "Churn Value" / "Churn"  (binary)
CLS_DROP   = []          # ids / leakage columns to remove e.g. ["CustomerID", "Churn Label", "Churn Score", "Churn Reason"]
PCA_VARIANCE = 0.90      # keep enough components to explain this much variance
# ---- Part C : regularised regression data ----
REG_PATH   = None        # "Housing.csv"  (None = sklearn diabetes demo)
REG_TARGET = "target"    # numeric column to predict e.g. "price"
REG_DROP   = []
OPT_LAMBDA = 1.0         # lambda (alpha) used for the final model comparison (pick from the error-vs-lambda plot)
# -------------------------------------------------------------------------

def load_any(path):
    return pd.read_excel(path) if path.endswith((".xlsx", ".xls")) else pd.read_csv(path)

def prepare(df, target, drop, stratify=False):
    """clean -> encode -> split -> scale. Returns X_train, X_test, y_train, y_test (scaled numeric columns)."""
    df = df.drop_duplicates().drop(columns=[c for c in drop if c in df.columns])
    for c in df.columns:                                              # numbers stored as text (e.g. ' ' in Total Charges)
        if df[c].dtype == object or str(df[c].dtype) == "str":
            conv = pd.to_numeric(df[c], errors="coerce")
            if conv.notna().mean() > 0.9: df[c] = conv
    for c in df.select_dtypes("number").columns: df[c] = df[c].fillna(df[c].median())   # missing numeric -> median
    y = df[target]; X = df.drop(columns=target)
    if y.dtype == object or str(y.dtype) == "str": y = pd.Series(LabelEncoder().fit_transform(y), index=y.index)   # 'Yes'/'No' -> 1/0
    X = X.fillna("Missing")
    num_cols = X.select_dtypes("number").columns.tolist()
    X = pd.get_dummies(X, drop_first=True).astype(float)                                 # categorical -> dummy columns
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if stratify else None)
    sc = StandardScaler(); Xtr[num_cols] = sc.fit_transform(Xtr[num_cols]); Xte[num_cols] = sc.transform(Xte[num_cols])   # fit on train only
    return Xtr, Xte, ytr, yte

# =============================== PART A: CLASSIFICATION ===============================
if CLS_PATH is None:
    from sklearn.datasets import load_breast_cancer
    raw = load_breast_cancer(as_frame=True).frame.rename(columns={"target": CLS_TARGET})
else:
    raw = load_any(CLS_PATH)

# EDA plots: target distribution + two important features (edit feature names if you want specific ones)
num_feats = raw.select_dtypes("number").columns.drop(CLS_TARGET).tolist()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x=CLS_TARGET, data=raw, ax=ax[0]); ax[0].set_title("Target distribution")
sns.kdeplot(data=raw, x=num_feats[0], hue=CLS_TARGET, common_norm=False, fill=True, ax=ax[1]); ax[1].set_title(f"{num_feats[0]} by target")
sns.boxplot(data=raw, x=CLS_TARGET, y=num_feats[1], ax=ax[2]); ax[2].set_title(f"{num_feats[1]} by target")
plt.tight_layout(); plt.show()

X_train, X_test, y_train, y_test = prepare(raw, CLS_TARGET, CLS_DROP, stratify=True)
print("Train:", X_train.shape, "Test:", X_test.shape)

def get_models():                                        # fresh models each time (>= 5 algorithms)
    return {"Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
            "Decision Tree": DecisionTreeClassifier(random_state=42),
            "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
            "Gradient Boosting": GradientBoostingClassifier(random_state=42),
            "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5)}

def run_all(Xtr, Xte, set_name, n_feats):
    """fit every model, return list of metric dicts + stored probabilities/predictions."""
    out, store = [], {}
    for name, m in get_models().items():
        t0 = time.time(); m.fit(Xtr, y_train); t = time.time() - t0
        pred, prob = m.predict(Xte), m.predict_proba(Xte)[:, 1]
        store[name] = (pred, prob)
        out.append({"Feature Set": set_name, "Model": name, "Features": n_feats, "Accuracy": accuracy_score(y_test, pred),
                    "Precision": precision_score(y_test, pred), "Recall": recall_score(y_test, pred),
                    "F1": f1_score(y_test, pred), "ROC-AUC": roc_auc_score(y_test, prob), "Train Time (s)": t})
    return out, store

res_A, store_A = run_all(X_train, X_test, "Original", X_train.shape[1])
metrics_A = pd.DataFrame(res_A).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
print(metrics_A.drop(columns="Feature Set").round(4).to_string(index=False))

fig, ax = plt.subplots(1, 5, figsize=(22, 4))                          # confusion matrices
for a, (n, (p, _)) in zip(ax, store_A.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, p)).plot(ax=a, cmap="Blues", colorbar=False); a.set_title(n, fontsize=9)
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(1, 2, figsize=(14, 5))                          # ROC + Precision-Recall curves
for n, (_, pr) in store_A.items():
    f, t, _ = roc_curve(y_test, pr); ax[0].plot(f, t, label=f"{n} ({roc_auc_score(y_test, pr):.3f})")
    p_, r_, _ = precision_recall_curve(y_test, pr); ax[1].plot(r_, p_, label=n)
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set(title="ROC curves", xlabel="FPR", ylabel="TPR"); ax[0].legend()
ax[1].set(title="Precision-Recall curves", xlabel="Recall", ylabel="Precision"); ax[1].legend(); plt.show()
print("BEST MODEL (by ROC-AUC):", metrics_A.loc[0, "Model"], round(metrics_A.loc[0, "ROC-AUC"], 4))
# For imbalanced data (like churn) also look at Recall / F1 / PR-curve, not only Accuracy.

# =============================== PART B: PCA and SVD ===============================
cum = np.cumsum(PCA().fit(X_train).explained_variance_ratio_)
k = int(np.argmax(cum >= PCA_VARIANCE) + 1)                            # components needed for PCA_VARIANCE
plt.figure(figsize=(7, 4)); plt.plot(range(1, len(cum) + 1), cum, "o--"); plt.axhline(PCA_VARIANCE, color="r", ls=":")
plt.axvline(k, color="g", ls=":", label=f"{k} components"); plt.xlabel("Components"); plt.ylabel("Cumulative variance"); plt.legend(); plt.show()
pca = PCA(n_components=k, random_state=42); Xtr_p, Xte_p = pca.fit_transform(X_train), pca.transform(X_test)
svd = TruncatedSVD(n_components=k, random_state=42); Xtr_s, Xte_s = svd.fit_transform(X_train), svd.transform(X_test)
# (PCA centers the data first; TruncatedSVD does not -> results are close but not identical.)

res_p, store_p = run_all(Xtr_p, Xte_p, "PCA", k)
res_s, store_s = run_all(Xtr_s, Xte_s, "SVD", k)
all_res = pd.DataFrame(res_A + res_p + res_s)
print(all_res.round(4).to_string(index=False))                         # comparative table: Original vs PCA vs SVD

fig, ax = plt.subplots(2, 3, figsize=(18, 9)); ax = ax.ravel()
for a, m_ in zip(ax, ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Train Time (s)"]):
    sns.barplot(data=all_res, x="Model", y=m_, hue="Feature Set", ax=a); a.set_title(m_); a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()
summ = all_res.groupby("Feature Set")[["Accuracy", "F1", "ROC-AUC", "Train Time (s)"]].mean().round(4)
print(summ)
d = summ.loc["PCA", "ROC-AUC"] - summ.loc["Original", "ROC-AUC"]
print(f"Conclusion: PCA ROC-AUC change vs original = {d:+.4f} using {k} of {X_train.shape[1]} features ->",
      "reduction helps" if d > 0.002 else "reduction hurts" if d < -0.002 else "performance about the same, but fewer features")

# =============================== PART C: RIDGE / LASSO / ELASTIC NET ===============================
rdf = load_any(REG_PATH) if REG_PATH else __import__("sklearn.datasets", fromlist=["x"]).load_diabetes(as_frame=True).frame.rename(columns={"target": REG_TARGET})
for c in rdf.columns:                                                  # yes/no columns -> 1/0
    if set(rdf[c].dropna().unique()) <= {"yes", "no"}: rdf[c] = rdf[c].map({"yes": 1, "no": 0})
Xr_tr, Xr_te, yr_tr, yr_te = prepare(rdf, REG_TARGET, REG_DROP)

def metrics(y, p):
    mse = mean_squared_error(y, p); return mse, np.sqrt(mse), mean_absolute_error(y, p), r2_score(y, p)
lin = LinearRegression().fit(Xr_tr, yr_tr)
print("Linear Regression  MSE/RMSE/MAE/R2:", [round(v, 4) for v in metrics(yr_te, lin.predict(Xr_te))])

lambdas = np.logspace(-3, 3, 50)                                       # many lambda (alpha) values
mk = {"Ridge": lambda a: Ridge(alpha=a), "Lasso": lambda a: Lasso(alpha=a, max_iter=10000),
      "ElasticNet": lambda a: ElasticNet(alpha=a, l1_ratio=0.5, max_iter=10000)}
rows, curve = [], {n: ([], []) for n in mk}
for n, f in mk.items():
    for a in lambdas:
        m = f(a).fit(Xr_tr, yr_tr)
        tr_rmse, te = np.sqrt(mean_squared_error(yr_tr, m.predict(Xr_tr))), metrics(yr_te, m.predict(Xr_te))
        curve[n][0].append(tr_rmse); curve[n][1].append(te[1])
        rows.append({"Model": n, "lambda": a, "Train RMSE": tr_rmse, "Test MSE": te[0], "Test RMSE": te[1], "Test MAE": te[2], "Test R2": te[3]})
lam_table = pd.DataFrame(rows)                                         # metrics for every lambda
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, n in zip(ax, mk):
    a.plot(lambdas, curve[n][0], label="Train RMSE"); a.plot(lambdas, curve[n][1], "--", label="Test RMSE")
    a.set_xscale("log"); a.set(title=n, xlabel="lambda", ylabel="RMSE"); a.legend()
plt.tight_layout(); plt.show()

final = {"Linear": lin, f"Ridge ({OPT_LAMBDA})": Ridge(alpha=OPT_LAMBDA).fit(Xr_tr, yr_tr),
         f"Lasso ({OPT_LAMBDA})": Lasso(alpha=OPT_LAMBDA, max_iter=10000).fit(Xr_tr, yr_tr),
         f"ElasticNet ({OPT_LAMBDA})": ElasticNet(alpha=OPT_LAMBDA, l1_ratio=0.5, max_iter=10000).fit(Xr_tr, yr_tr)}
tab = pd.DataFrame([dict(zip(["Model", "MSE", "RMSE", "MAE", "R2"], [n, *metrics(yr_te, m.predict(Xr_te))])) for n, m in final.items()])
print(tab.round(4).to_string(index=False))
coef = pd.DataFrame({n: m.coef_ for n, m in final.items()}, index=Xr_tr.columns).reset_index().melt("index", var_name="Model", value_name="Coefficient")
plt.figure(figsize=(14, 5)); sns.barplot(data=coef, x="index", y="Coefficient", hue="Model"); plt.xticks(rotation=45, ha="right"); plt.axhline(0, color="k", lw=.8); plt.show()
print("Zero coefficients -> Lasso:", int((final[f'Lasso ({OPT_LAMBDA})'].coef_ == 0).sum()), "| ElasticNet:", int((final[f'ElasticNet ({OPT_LAMBDA})'].coef_ == 0).sum()))
# REPORT: Ridge shrinks coefficients (never exactly 0); Lasso sets some to exactly 0 (feature selection); ElasticNet mixes both.
# Very large lambda -> underfitting (both errors rise); very small lambda -> same as plain linear regression.

In [ ]:
# =====================================================================
# ASSIGNMENT 4 (GENERIC): t-tests  (independent two-sample, paired, and the difference between them)
# HOW TO USE: paste your lists in STEP 0 (or load two columns from a CSV). Everything else is automatic.
# =====================================================================
import numpy as np
import pandas as pd
from scipy import stats

# ---------------------- STEP 0: CONFIG (EDIT THIS) ----------------------
ALPHA = 0.05
# Q1: INDEPENDENT groups (different people in each group, e.g. method A vs method B)
group_A = [65, 70, 68, 72, 66, 75, 69, 71, 67, 70]
group_B = [72, 75, 78, 74, 80, 77, 73, 79, 76, 81]
# Q2: PAIRED data (the SAME people measured twice, e.g. before vs after training) -> lists must be same length & same order
before = [52, 48, 65, 60, 55, 70, 58, 62, 50, 67]
after  = [60, 55, 70, 66, 63, 75, 64, 68, 57, 72]
# Q3: your own dataset to compare both tests on (same people, before/after). To read from a CSV instead, use e.g.
#   d = pd.read_csv("data.csv"); demo_before, demo_after = d["col1"].dropna().tolist(), d["col2"].dropna().tolist()
demo_before = [65, 72, 58, 80, 62, 70, 75, 68, 59, 74, 66, 61]
demo_after  = [69, 73, 62, 81, 64, 75, 76, 71, 62, 75, 67, 65]
# -------------------------------------------------------------------------

def independent_t(a, b, alpha=ALPHA, title="INDEPENDENT TWO-SAMPLE T-TEST"):
    """H0: mean_a = mean_b.  Pooled-variance t-test (assumes equal variances). Manual formula + scipy check."""
    a, b = np.array(a, float), np.array(b, float); n1, n2 = len(a), len(b)
    v1, v2 = a.var(ddof=1), b.var(ddof=1)                          # sample variances (ddof=1)
    dfree = n1 + n2 - 2
    sp = np.sqrt(((n1 - 1) * v1 + (n2 - 1) * v2) / dfree)          # pooled standard deviation
    t = (a.mean() - b.mean()) / (sp * np.sqrt(1 / n1 + 1 / n2))    # t = (mean1 - mean2) / SE
    p = 2 * (1 - stats.t.cdf(abs(t), df=dfree))                    # two-tailed p-value
    ts, ps = stats.ttest_ind(a, b)                                 # scipy check (should match)
    print(f"===== {title} =====")
    print(f"Mean A = {a.mean():.3f}, Mean B = {b.mean():.3f}, df = {dfree}")
    print(f"t-statistic = {t:.4f}, p-value = {p:.6f}   (scipy: t = {ts:.4f}, p = {ps:.6f})")
    print("Conclusion:", "Reject H0 -> significant difference between the groups." if p < alpha
          else "Fail to reject H0 -> no significant difference.", "\n")
    return t, p

def paired_t(before, after, alpha=ALPHA, title="PAIRED T-TEST"):
    """H0: mean difference (after - before) = 0.  Works on the per-person differences."""
    d = np.array(after, float) - np.array(before, float); n = len(d)
    se = d.std(ddof=1) / np.sqrt(n)                                # standard error of the mean difference
    t = d.mean() / se; dfree = n - 1
    p = 2 * (1 - stats.t.cdf(abs(t), df=dfree))
    ts, ps = stats.ttest_rel(after, before)                        # scipy check
    print(f"===== {title} =====")
    print(f"Mean difference = {d.mean():.3f}, SD of differences = {d.std(ddof=1):.3f}, df = {dfree}")
    print(f"t-statistic = {t:.4f}, p-value = {p:.6f}   (scipy: t = {ts:.4f}, p = {ps:.6f})")
    print("Conclusion:", "Reject H0 -> significant change after the treatment." if p < alpha
          else "Fail to reject H0 -> no significant change.", "\n")
    return t, p

# ---------------------- Q1: independent t-test ----------------------
independent_t(group_A, group_B, title="Q1: INDEPENDENT T-TEST (two teaching methods)")

# ---------------------- Q2: paired t-test ----------------------
paired_t(before, after, title="Q2: PAIRED T-TEST (before vs after training)")

# ---------------------- Q3: independent vs paired on the SAME data ----------------------
t_i, p_i = independent_t(demo_before, demo_after, title="Q3: INDEPENDENT T-TEST on before/after data")
t_p, p_p = paired_t(demo_before, demo_after, title="Q3: PAIRED T-TEST on before/after data")
print("Correlation between before & after:", round(np.corrcoef(demo_before, demo_after)[0, 1], 3))
print(f"Independent: t={t_i:.3f}, p={p_i:.4f}  |  Paired: t={t_p:.3f}, p={p_p:.4f}")
# WHY THEY DIFFER (write this in the answer):
# - Independent test treats the two lists as unrelated people, so the person-to-person spread (large variance) sits in the denominator.
# - Paired test uses each person's own difference, which removes the person-to-person variation -> much smaller standard error.
# - When the same subjects are measured twice (high correlation), the paired t-statistic is larger and the p-value smaller,
#   so it detects a real change that the independent test can miss. Use paired only for matched/same-subject data.